# 6A. Baseline

Baseline buat pembanding semua model di Fase 6.

- Regresi / time series: **Seasonal Naive** = `lag_168` (pickup jam yang sama minggu lalu).
- Klasifikasi: **kelas mayoritas** dan **kelas dari `lag_168`**.

Metrik pakai `script/tools/metrics.py`. Data dari `features_dataset.parquet`.

# 0. Import

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

sys.path.append("tools")
import metrics

# 1. Muat Dataset Fitur

In [ ]:
data = pd.read_parquet("../datasets/interim/2026/features_dataset.parquet")
data["hour"] = pd.to_datetime(data["hour"])

train = data[data["split"] == "train"]
val = data[data["split"] == "validation"]
test = data[data["split"] == "test"]

print(data.shape)
print(data["split"].value_counts())

# 2. Baseline Regresi - Seasonal Naive (`lag_168`)

In [ ]:
# pembagi MASE dihitung dari training: |aktual - lag168|
scale = metrics.mase_scale(train["target_pickup_count"], train["lag_168"])
print("MASE scale (train):", round(scale, 3))

rows = []
for nama, d in [("validation", val), ("test", test)]:
    pred = d["lag_168"].to_numpy()
    m = metrics.regression_metrics(d["target_pickup_count"], pred, scale=scale)
    rows.append({"periode": nama, **{k: round(v, 4) for k, v in m.items()}})

baseline_reg = pd.DataFrame(rows)
display(baseline_reg)

# 3. Baseline Klasifikasi - Mayoritas & `lag_168`

In [ ]:
LABELS = ["Rendah", "Sedang", "Tinggi"]
q1, q2 = train["target_pickup_count"].quantile([1 / 3, 2 / 3])
edges = [-np.inf, q1, q2, np.inf]
mayor = train["demand_level"].astype(str).mode().iloc[0]
print(f"Batas kelas: Rendah <= {q1:.2f}, Sedang <= {q2:.2f}, Tinggi > {q2:.2f}")
print("Kelas mayoritas (train):", mayor)

rows = []
for nama, d in [("validation", val), ("test", test)]:
    y_true = d["demand_level"].astype(str)
    pred_mayor = pd.Series([mayor] * len(d), index=d.index)
    pred_lag = pd.cut(d["lag_168"], edges, labels=LABELS).astype(str)

    for model, pred in [("majority", pred_mayor), ("lag_168", pred_lag)]:
        m = metrics.classification_metrics(y_true, pred, labels=LABELS)
        rows.append({
            "periode": nama, "model": model,
            "accuracy": round(m["accuracy"], 4),
            "precision_macro": round(m["precision_macro"], 4),
            "recall_macro": round(m["recall_macro"], 4),
            "f1_macro": round(m["f1_macro"], 4),
        })

baseline_clf = pd.DataFrame(rows)
display(baseline_clf)

# 4. Simpan Prediksi Baseline

In [ ]:
os.makedirs("../out/output_eda", exist_ok=True)
baseline_reg.to_csv("../out/output_eda/06_baseline_regression.csv", index=False)
baseline_clf.to_csv("../out/output_eda/06_baseline_classification.csv", index=False)

for nama, d in [("validation", val), ("test", test)]:
    out = d[["hour", "zone_id", "target_pickup_count", "lag_168", "demand_level"]].copy()
    out["pred_reg"] = d["lag_168"].to_numpy()
    out["pred_class_majority"] = mayor
    out["pred_class_lag_168"] = pd.cut(d["lag_168"], edges, labels=LABELS).astype(str)
    out.to_parquet(f"../out/output_eda/06_baseline_predictions_{nama}.parquet", index=False)

print("Tersimpan di out/output_eda/:")
print("- 06_baseline_regression.csv")
print("- 06_baseline_classification.csv")
print("- 06_baseline_predictions_validation.parquet")
print("- 06_baseline_predictions_test.parquet")

# 5. Confusion Matrix (lag_168, test)

In [ ]:
pred_lag_test = pd.cut(test["lag_168"], edges, labels=LABELS).astype(str)
m = metrics.classification_metrics(test["demand_level"].astype(str), pred_lag_test, labels=LABELS)
cm = pd.DataFrame(m["confusion"], index=[f"aktual_{l}" for l in m["labels"]],
                  columns=[f"pred_{l}" for l in m["labels"]])
display(cm)

## Catatan

- Untuk regresi/time series, prediksi baseline = `lag_168` (pickup minggu lalu jam yang sama).
- MASE pakai pembagi dari training: rata-rata |aktual - lag_168| di data training.
- Baseline ini dipakai sebagai pembanding wajib di Fase 6B/6C/6D.